# Baseline reproduction on a GPU — Gahoi et al. (2022), MixMT Subtask-1

**Linear 298-52.** The project's quality gate: until the harness reproduces a published
number, no model result can be interpreted (Workbook 2, section 4.4). Run this on Colab with
a GPU runtime (*Runtime → Change runtime type → T4 / L4 / A100*), then commit what it writes.

The recipe, and every place it had to assume something the paper does not say, is in
`docs/baseline/gahoi_reproduction.md`. Read that first: the comparison with the paper is
against Table 2's **automatic-transliteration row (ROUGE-L 0.48376, WER 0.72561)**, because
the dictionary behind the 0.61667 row was never released.

### What this writes, and what to commit
| Path | Commit? |
|---|---|
| `runs/*baseline-gahoi*.json` | **yes** — config, assumptions, GPU-hours |
| `results/results.jsonl` | **yes** — scored rows, raw and transliterated |
| `docs/COMPUTE_BUDGET.md` | **yes** — add one line per run |
| `out/baseline/*.txt` | no — model output; regenerable |
| checkpoints | **never** — 2.4 GB each |

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime -> Change runtime type -> GPU, then run again.")
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")

## 2. Pick a profile

`smoke` proves the path in minutes on 64 records and 30 steps; its numbers mean nothing.
`full` is the paper's 10,000 steps. Its cost on a T4 has **not been measured yet** — the
first full run measures it and the run record keeps the GPU-hours. mBART-large is a full
fine-tune of ~610M parameters; it fits a 16 GB T4 in fp16, and `--gradient-checkpointing`
is the fallback if it does not.

Seeds: the project reports three (13, 42, 1337). Run seed 13 for both conditions first;
add the other two once one full run has shown its cost.

In [ ]:
PROFILE    = "smoke"                 # "smoke" | "full"
CONDITIONS = ["both", "english"]     # the paper's two-input condition, then English-only
SEEDS      = [13]                    # -> [13, 42, 1337] for the reported number

CFG = {"smoke": dict(limit=64, max_steps=30, warmup=5),
       "full":  dict(limit=None, max_steps=10000, warmup=2500)}[PROFILE]
print(PROFILE, CFG, CONDITIONS, SEEDS)

In [ ]:
# Keep finished runs on Drive so a disconnect does not cost hours.
PERSIST_DIR = None      # e.g. "/content/drive/MyDrive/lrcs_baseline" for the full run
import pathlib
if PERSIST_DIR:
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST = pathlib.Path(PERSIST_DIR); PERSIST.mkdir(parents=True, exist_ok=True)
else:
    PERSIST = None
print("persisting to", PERSIST or "nowhere -- a disconnect loses this session's work")

## 3. Clone and install

In [ ]:
import os, subprocess, pathlib, shutil
REPO, BRANCH = ("https://github.com/298A-Team-2-Topic-23/Low-Resource-and-Code-Switched-Language-Systems.git",
                "main")
ROOT = pathlib.Path("/content/lrcs")
if ROOT.exists():
    shutil.rmtree(ROOT)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, str(ROOT)], check=True)
os.chdir(ROOT)
for f in ("models/train_mbart_baseline.py", "models/transliterate.py", "evaluation/run_eval.py"):
    assert pathlib.Path(f).exists(), f"{f} missing -- wrong branch?"
print("HEAD", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout)

In [ ]:
# transformers is held on 4.x, as for Models 1 and 2.
%pip install -q "transformers==4.57.6" "accelerate==1.10.1" sentencepiece protobuf sacrebleu==2.6.0 \
                indic-transliteration datasketch
import transformers
assert transformers.__version__.startswith("4."), "Runtime -> Restart session, then run from the top"
print("transformers", transformers.__version__)

## 4. The corpus and the splits

Upload `HinGE.pkl` (or `hinge.csv`). The baseline cannot use the synthetic corpus: its
two-input condition needs the Hindi column, which the synthetic generator leaves empty.
The split hashes must match the committed manifest before anything is trained.

In [ ]:
from google.colab import files
raw = pathlib.Path("data/raw"); raw.mkdir(parents=True, exist_ok=True)
for name in files.upload():
    shutil.move(name, raw / name)
if not (raw / "hinge.csv").exists():
    subprocess.run(["python", "scripts/convert_hinge_pkl.py", "data/raw/HinGE.pkl", "data/raw/hinge.csv",
                    "--expected-sha256", "e78c8f48af3eea0b141e767082189b7b619e9cbe50ad8f0b2a4863b82ebc5960"], check=True)
r = subprocess.run(["python", "scripts/run_pipeline.py", "--input", "data/raw/hinge.csv", "--seed", "42"],
                   capture_output=True, text=True)
print(r.stdout[-1200:]); assert r.returncode == 0, r.stderr[-800:]

import json, hashlib
man = json.load(open("data/processed/manifests/hinge_split_manifest.json"))
for name, e in man["splits"].items():
    same = hashlib.sha256(pathlib.Path(e["file"]).read_bytes()).hexdigest() == e["sha256"]
    print(f"  {name:5s} {e['records']:>5}  {'MATCH' if same else 'DIFFERS'}")
    assert same, "splits differ from the committed manifest -- stop and investigate"

## 5. Test set

The paper scored the **500-sentence MixMT Subtask-1 test set**. If you have its references,
upload a TSV with `english`, `hindi` and `hinglish` columns and set `MIXMT_TSV`. Otherwise
the HinGE test split is used, and the comparison with Table 2 is indicative only: a
different test set, and the paper also trained on HinGE's synthetic part.

In [ ]:
MIXMT_TSV = None          # e.g. "data/raw/mixmt_test.tsv" after uploading it
TEST = MIXMT_TSV or "data/processed/hinge/test.tsv"
TEST_NAME = "MixMT Subtask-1" if MIXMT_TSV else "HinGE test split"
print("scoring on", TEST_NAME, "->", TEST)

## 6. Fine-tune and decode

In [ ]:
OUT = pathlib.Path("out/baseline")
for cond in CONDITIONS:
    for seed in SEEDS:
        raw_out = OUT / f"{cond}.s{seed}.raw.txt"
        kept = PERSIST / raw_out.name if PERSIST else None
        if kept and kept.exists():
            OUT.mkdir(parents=True, exist_ok=True); shutil.copy(kept, raw_out)
            shutil.copy(PERSIST / "test.ref.txt", OUT / "test.ref.txt")
            print("already finished:", raw_out.name); continue
        cmd = ["python", "models/train_mbart_baseline.py", "--condition", cond,
               "--train", "data/processed/hinge/train.tsv", "--dev", "data/processed/hinge/dev.tsv",
               "--test", TEST, "--out", str(OUT), "--seed", str(seed),
               "--max-steps", str(CFG["max_steps"]), "--warmup-steps", str(CFG["warmup"])]
        if CFG["limit"]:
            cmd += ["--limit", str(CFG["limit"])]
        subprocess.run(cmd, check=True)
        shutil.rmtree(OUT / f"{cond}.s{seed}.ckpt", ignore_errors=True)   # never commit checkpoints
        if PERSIST:
            shutil.copy(raw_out, PERSIST / raw_out.name); shutil.copy(OUT / "test.ref.txt", PERSIST)

## 7. Transliterate (Table 2, row 2)

The paper used the `indicate` library. Its 2022 release (0.1.0) needs TensorFlow and
tensorflow-addons; the cell tries it first. If it cannot be installed, it falls back to
`sanscript` and **labels the result a deviation** — that number is not the paper's row 2.

In [ ]:
r = subprocess.run(["pip", "install", "-q", "indicate==0.1.0"], capture_output=True, text=True)
try:
    from indicate import hindi2english; hindi2english("घर"); BACKEND = "indicate"
except Exception as exc:
    BACKEND = "sanscript"
    print("indicate 0.1.0 unavailable here:", repr(exc)[:200])
print("transliteration backend:", BACKEND, "" if BACKEND == "indicate" else "(DEVIATION from the paper)")
for f in sorted(OUT.glob("*.raw.txt")):
    subprocess.run(["python", "models/transliterate.py", str(f), str(f).replace(".raw.", ".auto."),
                    "--backend", BACKEND], check=True)

## 8. Score against Table 2

Each condition is scored twice, as in the paper: raw output (row 1, 0.39091 / 0.81884) and
after automatic transliteration (row 2, 0.48376 / 0.72561). `--check-baseline` compares with
the 0.617 row, which this reproduction is **not expected** to reach; read its verdict next
to row 2, as `docs/baseline/gahoi_reproduction.md` explains.

In [ ]:
label = "SMOKE, not a result" if PROFILE == "smoke" else TEST_NAME
for cond in CONDITIONS:
    for stage in ("raw", "auto"):
        hyps = [str(OUT / f"{cond}.s{s}.{stage}.txt") for s in SEEDS]
        system = f"Gahoi repro, {cond}, {stage}" + ("" if stage == "raw" else f" ({BACKEND})") + f" [{label}]"
        subprocess.run(["python", "evaluation/run_eval.py", "--hyp", *hyps, "--ref", str(OUT / "test.ref.txt"),
                        "--system", system, "--check-baseline", "--out", "results/results.jsonl"], check=True)
subprocess.run(["python", "evaluation/run_eval.py", "--report", "results/results.jsonl"], check=True)

## 9. Commit

Commit `runs/*baseline-gahoi*.json`, `results/results.jsonl`, and one line per run in
`docs/COMPUTE_BUDGET.md`. A smoke run's records are evidence the path works; label them so.
Then fill the result table in `docs/baseline/gahoi_reproduction.md` — including a miss,
which is a reportable finding, not a failure to hide.